# FLAG 2027 - Notebook 41 RUN-ALL: train the FACE encoder for voice matching (CLIP ViT-L/14 + LoRA, cross-video positives)

Research round 9 (`research/round9_2026-11-05.md`): the systems that beat the frozen-feature ceiling trained their encoders cross-modally on thousands of identities (FAME 2024 winner: both encoders trainable, lr 1e-5; 6 points from that stage). NB40 does the voice side (ECAPA). This notebook does the face side, which we have never trained: our face input is one frozen CLIP ViT-L/14 vector of the whole image.

- model: CLIP ViT-L/14 vision tower with **LoRA** (r 8, alpha 16, on the query and value projections of all 24 blocks; about 1.6M trainable weights), the pretrained projection kept, so the output lives in the same 768-d space as our stored CLIP features. fp16 autocast, gradient checkpointing, 12 identities x 4 clips per step.
- target: the cached voice stack of system E (ECAPA + WavLM-sv + ResNet293 + ReDimNet2 + audeering + Whisper, l2 per block) through a small projection; no audio is decoded here.
- loss: SupCon L_out between the projected LoRA-CLIP face embedding and the projected voice vector, plus an AAM identity loss on the 768-d face embedding whose class weights start at the frozen per-identity means (keeps what CLIP knows). Positives are all same-identity rows in the batch, which come from different clips and videos (the cross-video pairing that gave a FAME 2024 team 3 points).
- data: VoxCeleb2 parts `vox2_test_mp4` + `vox2_mp4_1` rebuilt row for row (NB07 rule, Bengali removed) with **five frames per clip** (the stored 1.0-s frame plus frames at 0, 2, 3, 4 s; a random one per step), MAV-Celeb v1/v3/extra faces (one per row). About 3,300 identities. **Challenge faces are never trained on**; they are reported during training and used only in the CV afterwards. Checkpoint selection on 10% held-out external identities. Checkpoints on Drive, resumable.
- then: the LoRA-CLIP embedding of every row's canonical face (challenge faces, MAV row faces, VoxCeleb 1.0-s frames) is extracted and tested in system E's recipe on the usual CV, next to the frozen CLIP (`F_cft_add`) and replacing it (`F_cft_rep`). Same rule as always. If adopted, NB20 gets system J.

Rules: an external image encoder fine-tuned on VoxCeleb2 and MAV-Celeb faces, declared; no test data anywhere; per-trial scoring.

**How to run:** T4 GPU, `HF_TOKEN` in Colab Secrets, Runtime > Run all. About 4.5-5 h (download ~40 min, frames ~30 min, MAV faces ~10 min, training ~2.5 h, extraction ~15 min, CV ~25 min). If the session dies, Run all again: data is rebuilt, training resumes from the Drive checkpoint, finished stages are skipped. Paste the **SUMMARY**.


In [ ]:
# Cell 1 [NB41 v1 5-Nov] - Drive, GPU (hard check), packages (peft needs Colab's old torchao removed, as in NB16), HF token (Colab Secrets only)
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q uninstall -y torchao
!pip -q install peft speechbrain huggingface_hub
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run again.'
dev = 'cuda'
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found in Colab Secrets (key icon on the left): add it and enable notebook access, then run again.'
print('[NB41 v1 5-Nov] Device:', torch.cuda.get_device_name(0), '| HF token found: True | disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [NB41 v1 5-Nov] - face files in our row order (challenge train = report only; MAV v1/v3 + extra; VoxCeleb2 parts rebuilt with FIVE frames per clip),
#          the cached VOICE stack of system E for every row (the training target), the stored frozen CLIP rows (alignment check + AAM init)
import os, re, json, time, zipfile, subprocess, glob, copy
import numpy as np, pandas as pd, librosa, soundfile as sf
import torch.nn as nn, torch.nn.functional as Fn
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
FINAL = os.path.join(PROJECT, 'final'); os.makedirs(FINAL, exist_ok=True)
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
IMG = ('.jpg', '.jpeg', '.png')
# ---- challenge train: faces + stored features (report only, never trained on) ----
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
if not os.path.exists(os.path.join(FULL, 'train_set.zip.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(IMG + ('.txt', '.csv'))])
    open(os.path.join(FULL, 'train_set.zip.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def tok(tokens, exts, fallback):
    t = [x for x in tokens if x.lower().endswith(exts)]; return t[0] if len(t) == 1 else tokens[fallback]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
lp = find_one('train_English.txt'); lines = [l.split() for l in open(lp) if l.strip()]
FACE_CH = [resolve(tok(t, IMG, 3), lp) for t in lines]; assert None not in FACE_CH, 'challenge face files not found - tell Claude'
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int); assert len(y) == len(FACE_CH)
cls2spk = {int(l[5]): l[4] for l in lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
# ---- MAV v1/v3 faces (same rows as mavceleb_v1v3.npz) ----
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_f, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(fm, os.path.join(MAVD, ver))
            mav_f.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
# ---- MAV extra faces (the SAME random face per clip as the stored extraction: same rng(0) sequence as NB24/NB35/NB39/NB40) ----
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz'))); MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
extra_f = [r[1] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
# ---- VoxCeleb2 parts: rebuild the rows (NB07 rule + Bengali filter, as NB26/NB40), FIVE frames per clip: f0 = the stored 1.0-s frame, f1-f4 at 0/2/3/4 s ----
try:
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})
def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def language_of(paths, bs=32):
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        wavs = [librosa.load(paths[j], sr=16000, mono=True)[0][:160000] for j in idx]
        lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max()); b = torch.zeros(len(wavs), L)
        for k, w in enumerate(wavs):
            b[k, :len(w)] = torch.from_numpy(w)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), (lens / L).to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)
def ffmpeg_job(job):
    mp4, wav, stem = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    v = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '1.0', '-i', mp4, '-frames:v', '1', '-q:v', '2', stem + '_f0.jpg']).returncode
    for k, t in enumerate(['0.0', '2.0', '3.0', '4.0'], 1):
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', t, '-i', mp4, '-frames:v', '1', '-q:v', '2', '%s_f%d.jpg' % (stem, k)])
    return a == 0 and v == 0 and os.path.exists(wav) and os.path.exists(stem + '_f0.jpg')
ZIPS = [('vox2/vox2_test_mp4.zip', 40), ('vox2/vox2_mp4_1.zip', 20)]
VOX = {}
for zname, cpi in ZIPS:
    tag = os.path.basename(zname).replace('.zip', ''); OLD = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    W = '/content/vox_work/' + tag; os.makedirs(W, exist_ok=True); rows_local = os.path.join(W, 'rows.npz'); t0 = time.time()
    if os.path.exists(rows_local):
        r = np.load(rows_local); stems = [os.path.join(W, x) for x in r['stem']]; spk = r['spk']
    else:
        zpath = hf_hub_download('ProgramComputer/voxceleb', zname, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
        zf = zipfile.ZipFile(zpath); tree = {}
        for n in zf.namelist():
            mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
            if mm:
                tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
        rng = np.random.default_rng(0); picked = []
        for pid, vids in tree.items():
            pools = [list(rng.permutation(v)) for v in vids.values()]; chosen = []
            while len(chosen) < cpi and any(pools):
                for p in pools:
                    if p and len(chosen) < cpi:
                        chosen.append(p.pop())
            picked += [(pid, c) for c in chosen]
        jobs = []
        for i, (pid, member) in enumerate(picked):
            jobs.append((zf.extract(member, W + '/mp4'), '%s/%06d.wav' % (W, i), '%s/%06d' % (W, i)))
        zf.close(); os.remove(zpath)
        with ThreadPoolExecutor(8) as ex:
            ok = list(tqdm(ex.map(ffmpeg_job, jobs), total=len(jobs), desc='ffmpeg ' + tag))
        shutil.rmtree(W + '/mp4', ignore_errors=True)
        keep_ = [i for i, g in enumerate(ok) if g]
        wavs = [jobs[i][1] for i in keep_]; stems = [jobs[i][2] for i in keep_]; spk = np.array(['vox_' + picked[i][0] for i in keep_])
        lang = language_of(wavs); nb = lang != 'bn'
        stems = [s for s, k in zip(stems, nb) if k]; spk = spk[nb]
        for w in wavs:
            os.remove(w)                                   # audio was only needed for the language filter
        np.savez(rows_local, stem=np.array([os.path.basename(s) for s in stems]), spk=spk)
    assert len(spk) == len(OLD['spk']) and (spk == OLD['spk']).all(), '%s: row order differs from the stored features - tell Claude' % tag
    frames = [[s + '_f%d.jpg' % k for k in range(5) if os.path.exists(s + '_f%d.jpg' % k)] for s in stems]
    VOX[tag] = dict(frames=frames, spk=list(spk), clip=OLD['clip'])
    note('%s: %d rows, %d identities | frames per clip: mean %.2f | %.0f min' % (tag, len(spk), len(set(spk)), np.mean([len(f) for f in frames]), (time.time() - t0) / 60))
del LID; torch.cuda.empty_cache()
# ---- voice target = system E's voice stack (l2 per block, concatenated) for every row, from the cached feature files ----
VOICE = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']
CHF = {'ecapa': Xv}
for b in ['clip'] + VOICE[1:]:
    CHF[b] = np.load(os.path.join(FEATDIR, b + '.npz'))['train']
for b in VOICE[1:]:
    if b == 'wavlm_sv':
        continue
    d = np.load(os.path.join(FEATDIR, b + '.npz'))
    if b != 'r293' or 'mav' in d.files:
        MAV[b] = d['mav']
    if b != 'r293':
        EXTRA[b] = d['extra']
def vox_block(tag, b):
    pn, pz = os.path.join(FEATDIR, 'vox_%s_%s.npy' % (tag, b)), os.path.join(FEATDIR, 'vox_%s_%s.npz' % (tag, b))
    return np.load(pn) if os.path.exists(pn) else np.load(pz)['h']
for tag in VOX:
    d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    for b in VOICE:
        VOX[tag][b] = d[b] if b in d else vox_block(tag, b)
def vtab(src):
    return np.concatenate([l2(src[b]) for b in VOICE], 1).astype(np.float32)
TAGS = [os.path.basename(z).replace('.zip', '') for z, _ in ZIPS]
ALL_FACES = [[p] for p in FACE_CH] + [[p] for p in mav_f] + [[p] for p in extra_f] + sum([VOX[t]['frames'] for t in TAGS], [])
ALL_SPK = np.array(['ch_%d' % c for c in y] + list(MAV['spk']) + list(EXTRA['spk']) + sum([VOX[t]['spk'] for t in TAGS], []))
ALL_CLIP0 = np.concatenate([CHF['clip'], MAV['clip'], EXTRA['clip']] + [VOX[t]['clip'] for t in TAGS]).astype(np.float32)
ALL_VOICE = np.concatenate([vtab(CHF), vtab(MAV), vtab(EXTRA)] + [vtab(VOX[t]) for t in TAGS]).astype(np.float32)
N_CH, N_MAV, N_EX = len(y), len(MAV['spk']), len(EXTRA['spk'])
SEG = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_EX, int)]
for i, t in enumerate(TAGS):
    SEG = np.r_[SEG, (3 + i) * np.ones(len(VOX[t]['spk']), int)]
assert len(ALL_FACES) == len(ALL_SPK) == len(ALL_CLIP0) == len(ALL_VOICE) == len(SEG) and np.isfinite(ALL_VOICE).all()
note('rows: challenge %d | MAV v1/v3 %d | MAV extra %d | %s | total %d rows, %d identities | voice target %d-d' % (N_CH, N_MAV, N_EX, ' | '.join('%s %d' % (t, len(VOX[t]['spk'])) for t in TAGS), len(ALL_SPK), len(set(ALL_SPK)), ALL_VOICE.shape[1]))
# ---- frozen CLIP + preprocessing; alignment check: the canonical face of 100 random rows per split must reproduce the stored CLIP rows ----
from transformers import CLIPModel
import torchvision.transforms as T
CLIPM = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(dev).eval()
TF = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(), T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])
def load_img(p):
    return TF(Image.open(p).convert('RGB'))
POOL = ThreadPoolExecutor(8)
def batch_images(paths):
    return torch.stack(list(POOL.map(load_img, paths))).to(dev)
@torch.no_grad()
def clip_frozen(paths, bs=64):
    out = []
    for i in range(0, len(paths), bs):
        with torch.autocast('cuda', dtype=torch.float16):
            e = CLIPM.visual_projection(CLIPM.vision_model(pixel_values=batch_images(paths[i:i + bs])).pooler_output)
        out.append(e.float().cpu().numpy())
    return np.concatenate(out)
rng_c = np.random.default_rng(1)
for name, lo, hi in [('challenge', 0, N_CH), ('mav', N_CH, N_CH + N_MAV), ('extra', N_CH + N_MAV, N_CH + N_MAV + N_EX)] + [(t, int(np.where(SEG == 3 + i)[0][0]), int(np.where(SEG == 3 + i)[0][-1]) + 1) for i, t in enumerate(TAGS)]:
    idx = rng_c.choice(np.arange(lo, hi), 100, replace=False); new = clip_frozen([ALL_FACES[i][0] for i in idx])
    cos = (l2(new) * l2(ALL_CLIP0[idx])).sum(1); note('alignment faces %-14s median CLIP cosine %.4f (torchvision preprocessing vs stored CLIPProcessor rows)' % (name, np.median(cos)))
    assert np.median(cos) > 0.97, '%s: face files do not match the stored CLIP rows - tell Claude' % name


In [ ]:
# Cell 3 [NB41 v1 5-Nov] - LoRA on CLIP ViT-L/14 (q, v of all blocks), trained with SupCon L_out against the projected cached voice stack + AAM identity
#          loss on the 768-d face embedding (class weights start at the frozen per-identity CLIP means). External rows only; 10% of external identities
#          held out for checkpoint selection; challenge rows reported, never trained on. One random frame per clip per step. Resumable from Drive.
from sklearn.metrics import roc_curve
from peft import LoraConfig, inject_adapter_in_model
try:
    CLIPM.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False}); CLIPM.enable_input_require_grads()
except Exception as e:
    print('gradient checkpointing not enabled (%s) - batch may need to shrink' % repr(e)[:80])
for p in CLIPM.parameters():
    p.requires_grad = False
VIS = inject_adapter_in_model(LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=['q_proj', 'v_proj']), CLIPM.vision_model)
for n_, p in VIS.named_parameters():
    p.requires_grad = 'lora_' in n_
PROJ = CLIPM.visual_projection
n_tr = sum(p.numel() for p in VIS.parameters() if p.requires_grad); print('trainable LoRA weights: %.2f M' % (n_tr / 1e6)); assert n_tr > 0, 'LoRA layers were not injected - tell Claude'
EXT_ROWS = np.where(SEG > 0)[0]
ext_ids = sorted(set(ALL_SPK[EXT_ROWS])); rng0 = np.random.default_rng(0); perm = rng0.permutation(len(ext_ids)); n_val = int(round(0.1 * len(ext_ids)))
VAL_IDS = set(ext_ids[i] for i in perm[:n_val]); TR_IDS = sorted(ext_ids[i] for i in perm[n_val:])
tr_rows = np.array([i for i in EXT_ROWS if ALL_SPK[i] not in VAL_IDS]); val_rows = np.array([i for i in EXT_ROWS if ALL_SPK[i] in VAL_IDS])
cls_of = {s: i for i, s in enumerate(TR_IDS)}; ytr = np.array([cls_of[ALL_SPK[i]] for i in tr_rows]); pos_by_cls = [np.where(ytr == c)[0] for c in range(len(TR_IDS))]
note('training identities %d (%d rows) | held-out external identities %d (%d rows) | challenge rows %d: report only' % (len(TR_IDS), len(tr_rows), len(VAL_IDS), len(val_rows), N_CH))
class Proj(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, d))
    def forward(self, x):
        return self.net(x)
proj_f, proj_v = Proj(768).to(dev), Proj(ALL_VOICE.shape[1]).to(dev)
Wc = np.stack([l2(ALL_CLIP0[tr_rows[pos]]).mean(0) for pos in pos_by_cls]); W_ID = nn.Parameter(torch.tensor(l2(Wc), device=dev, dtype=torch.float32))
VOICE_T = torch.tensor(ALL_VOICE, device=dev)
def aam_loss(e, lab, m=0.2, s=30.0):
    cos = Fn.normalize(e, dim=1) @ Fn.normalize(W_ID, dim=1).t(); oh = Fn.one_hot(lab, cos.shape[1]).bool()
    theta = torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6)); logits = s * torch.where(oh, torch.cos(theta + m), cos)
    return Fn.cross_entropy(logits, lab)
def lout(a, b, lab, tau=0.07):
    S = Fn.normalize(a, dim=1) @ Fn.normalize(b, dim=1).t(); S = S / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def face_embed(x):
    with torch.autocast('cuda', dtype=torch.float16):
        e = PROJ(VIS(pixel_values=x).pooler_output)
    return e.float()
@torch.no_grad()
def embed_rows(rows, bs=64):
    VIS.eval(); out = []
    for i in range(0, len(rows), bs):
        out.append(face_embed(batch_images([ALL_FACES[r][0] for r in rows[i:i + bs]])))
    return torch.cat(out)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def pairs_of(rows, n, seed, gender=None):
    rng_ = np.random.default_rng(seed); spk_ = np.array(sorted(set(ALL_SPK[rows]))); by = {s: rows[ALL_SPK[rows] == s] for s in spk_}
    V, F, Lb = [], [], []
    for _ in range(n):
        s = rng_.choice(spk_); V.append(rng_.choice(by[s])); F.append(rng_.choice(by[s])); Lb.append(1)
        s = rng_.choice(spk_)
        cand = [t for t in spk_ if t != s and (gender is None or gender[t] == gender[s])] or [t for t in spk_ if t != s]
        t = rng_.choice(cand); V.append(rng_.choice(by[s])); F.append(rng_.choice(by[t])); Lb.append(0)
    return np.array(V), np.array(F), np.array(Lb)
CH_ROWS = np.arange(N_CH); ch_gender = {'ch_%d' % c: id_gender[c] for c in ids}
PAIRS = {'val_ext': pairs_of(val_rows, 10000, 0), 'ch_std': pairs_of(CH_ROWS, 2000, 0), 'ch_gender': pairs_of(CH_ROWS, 2000, 100, ch_gender)}
@torch.no_grad()
def evaluate():
    proj_f.eval(); proj_v.eval(); res = {}
    for name, (Vi, Fi, Lb) in PAIRS.items():
        frows = np.unique(Fi); pos = {int(r): i for i, r in enumerate(frows)}
        pf = Fn.normalize(proj_f(embed_rows(frows)), dim=1); pv = Fn.normalize(proj_v(VOICE_T[torch.tensor(Vi, device=dev)]), dim=1)
        i_f = torch.tensor([pos[int(r)] for r in Fi], device=dev)
        res[name] = eer(Lb, (pf[i_f] * pv).sum(1).cpu().numpy())
    proj_f.train(); proj_v.train(); return res
def fmt(r):
    return 'held-out external EER %.2f | challenge std %.2f gender %.2f' % (r['val_ext'], r['ch_std'], r['ch_gender'])
P_ID, K_CLIP = 12, 4
TOTAL, WARMUP, EVAL_EVERY, PATIENCE, LAM_ID = 8000, 200, 500, 6, 0.3   # v2 6-Nov: NB40 showed 4000/400/4/1.0 stops before the encoder moves
opt = torch.optim.AdamW([{'params': [p for p in VIS.parameters() if p.requires_grad], 'lr': 1e-4, 'weight_decay': 0.01},
                         {'params': list(proj_f.parameters()) + list(proj_v.parameters()), 'lr': 1e-3, 'weight_decay': 0.01},
                         {'params': [W_ID], 'lr': 1e-4, 'weight_decay': 0.0}])
scaler = torch.amp.GradScaler('cuda')
CKPT, BESTP = os.path.join(FINAL, 'clip_ft_ckpt.pt'), os.path.join(FINAL, 'clip_ft.pt')
CFG = dict(encoder='openai/clip-vit-large-patch14 + LoRA r8 alpha16 q,v', data='MAV v1/v3/extra faces + VoxCeleb2 vox2_test_mp4 + vox2_mp4_1 (5 frames/clip, bn removed), challenge rows excluded',
           loss='SupCon L_out tau 0.07 vs projected cached voice stack (ecapa+wavlm_sv+r293+redim+audeer+whisper) + AAM(m 0.2, s 30) identity loss', batch='12 ids x 4 clips', lr='LoRA 1e-4, projections 1e-3, class weights 1e-4', warmup=WARMUP, total=TOTAL)
def lora_state():
    return {k: v.detach().cpu() for k, v in VIS.state_dict().items() if 'lora_' in k}
step, best, best_step, wait, hist, ref = 0, 1e9, 0, 0, [], None
rng = np.random.default_rng(1); torch.manual_seed(1)
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=dev, weights_only=False)
    VIS.load_state_dict(ck['lora'], strict=False); proj_f.load_state_dict(ck['proj_f']); proj_v.load_state_dict(ck['proj_v']); W_ID.data.copy_(ck['w_id']); opt.load_state_dict(ck['opt'])
    step, best, best_step, wait, hist, ref = ck['step'], ck['best'], ck['best_step'], ck['wait'], ck['hist'], ck['ref']; rng = np.random.default_rng(1 + step)
    print('resumed from step %d (best held-out EER %.2f at step %d)' % (step, best, best_step))
def save_ckpt():
    torch.save({'lora': lora_state(), 'proj_f': proj_f.state_dict(), 'proj_v': proj_v.state_dict(), 'w_id': W_ID.data, 'opt': opt.state_dict(), 'step': step, 'best': best, 'best_step': best_step, 'wait': wait, 'hist': hist, 'ref': ref, 'cfg': CFG}, CKPT)
proj_f.train(); proj_v.train(); t0 = time.time(); n_done = 0; n_bad = 0
while step < TOTAL and wait < PATIENCE:
    chosen = rng.choice(len(TR_IDS), P_ID, replace=False)
    j = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]); rws = tr_rows[j]
    paths = [ALL_FACES[r][rng.integers(len(ALL_FACES[r]))] for r in rws]           # a random frame of the clip (VoxCeleb: 1 of up to 5)
    x = batch_images(paths); lab = torch.tensor(ytr[j], device=dev)
    if step < WARMUP:
        VIS.eval()
        with torch.no_grad():
            e = face_embed(x)                         # projections and class weights warm up on the frozen encoder
    else:
        VIS.train(); e = face_embed(x)
    pf = proj_f(e); pv = proj_v(VOICE_T[torch.tensor(rws, device=dev)])
    loss_al = lout(pf, pv, lab); loss_id = aam_loss(e, lab); loss = loss_al + LAM_ID * loss_id
    if not torch.isfinite(loss):
        n_bad += 1; opt.zero_grad(); assert n_bad < 30, 'training diverged (30 non-finite losses) - tell Claude'; continue
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    step += 1; n_done += 1
    if n_done == 20:
        print('  speed %.2f s/step -> remaining %d steps about %.0f min' % ((time.time() - t0) / 20, TOTAL - step, (TOTAL - step) * (time.time() - t0) / 20 / 60))
    if step == WARMUP:
        ref = evaluate(); note('step %5d REFERENCE frozen CLIP + warmed-up projections: %s' % (step, fmt(ref))); save_ckpt()
    elif step % EVAL_EVERY == 0 or step == TOTAL:
        r = evaluate(); improved = r['val_ext'] < best - 0.05
        hist.append((step, round(r['val_ext'], 2), round(r['ch_std'], 2), round(r['ch_gender'], 2), round(float(loss_al), 3), round(float(loss_id), 3)))
        if improved:
            best, best_step, wait = r['val_ext'], step, 0
            torch.save({'lora': lora_state(), 'proj_f': proj_f.state_dict(), 'proj_v': proj_v.state_dict(), 'step': step, 'val': r, 'ref': ref, 'cfg': CFG}, BESTP)
        else:
            wait += 1
        note('step %5d | loss align %.3f id %.3f | %s | %.0f min%s' % (step, float(loss_al), float(loss_id), fmt(r), (time.time() - t0) / 60, '  <- best, saved' if improved else ''))
        save_ckpt()
best_ck = torch.load(BESTP, map_location=dev, weights_only=False); VIS.load_state_dict(best_ck['lora'], strict=False); VIS.eval()
if ref is None:
    ref = best_ck.get('ref')
note('training done: best held-out external EER %.2f at step %d (reference at step %d: %.2f); challenge std %.2f -> %.2f, gender %.2f -> %.2f (projection scorer, not the final system)' % (
    best_ck['val']['val_ext'], best_ck['step'], WARMUP, ref['val_ext'] if ref else float('nan'), ref['ch_std'] if ref else float('nan'), best_ck['val']['ch_std'], ref['ch_gender'] if ref else float('nan'), best_ck['val']['ch_gender']))


In [ ]:
# Cell 4 [NB41 v1 5-Nov] - LoRA-CLIP embedding of every row's canonical face (challenge faces, MAV row faces, VoxCeleb 1.0-s frames) -> features/clip_ft.npz
#          (train/mav/extra) + features/vox_<tag>_clip_ft.npy; sanity: identity gap and how far the encoder moved. Skipped if the files exist.
OUT_NPZ = os.path.join(FEATDIR, 'clip_ft.npz'); VOX_OUT = {t: os.path.join(FEATDIR, 'vox_%s_clip_ft.npy' % t) for t in TAGS}
def enc_rows(lo, hi, desc):
    rows = np.arange(lo, hi); out = []
    for i in tqdm(range(0, len(rows), 64), desc=desc):
        out.append(embed_rows(rows[i:i + 64]).cpu().numpy())
    return np.concatenate(out).astype(np.float32)
if os.path.exists(OUT_NPZ) and all(os.path.exists(p) for p in VOX_OUT.values()):
    CFT = dict(np.load(OUT_NPZ)); print('LoRA-CLIP features already on Drive')
else:
    t0 = time.time()
    CFT = {'train': enc_rows(0, N_CH, 'clip_ft challenge'), 'mav': enc_rows(N_CH, N_CH + N_MAV, 'clip_ft mav'), 'extra': enc_rows(N_CH + N_MAV, N_CH + N_MAV + N_EX, 'clip_ft extra')}
    np.savez(OUT_NPZ, **CFT)
    for i, t in enumerate(TAGS):
        w = np.where(SEG == 3 + i)[0]; E = enc_rows(int(w[0]), int(w[-1]) + 1, 'clip_ft ' + t); assert len(E) == len(VOX[t]['spk']); np.save(VOX_OUT[t], E)
    note('LoRA-CLIP extracted for all rows in %.0f min -> clip_ft.npz + %s' % ((time.time() - t0) / 60, ', '.join(os.path.basename(p) for p in VOX_OUT.values())))
assert all(np.isfinite(CFT[s]).all() for s in CFT), 'non-finite LoRA-CLIP features - tell Claude'
def gap(E, S, n=600, seed=0):
    r = np.random.default_rng(seed); idx = r.choice(len(S), min(n, len(S)), replace=False); Z = l2(E[idx]); C = Z @ Z.T
    same = (S[idx][:, None] == S[idx][None, :]) & ~np.eye(len(idx), dtype=bool); diff = S[idx][:, None] != S[idx][None, :]
    return C[same].mean() - C[diff].mean()
note('identity gap (same - different cosine) on challenge rows: frozen CLIP %.3f | LoRA-CLIP %.3f' % (gap(CHF['clip'], y), gap(CFT['train'], y)))
note('frozen vs LoRA-CLIP, challenge rows: median cosine %.3f (1.0 = unchanged encoder)' % np.median((l2(CHF['clip']) * l2(CFT['train'])).sum(1)))
NOTES41 = SUMMARY
del CLIPM, VIS; torch.cuda.empty_cache()


In [ ]:
# Cell 5 - features, labels, CV folds, harness (NB36 Cell 2 verbatim, same as NB39 Cell 5)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# Cell 6 [NB41 v1 5-Nov] - CV: the LoRA-CLIP face block in system E's recipe (voice R4 + whisper, NAP-8; CE + 2*OPL + 0.5*L_out tau 0.07).
#          F_cft_add = frozen CLIP + LoRA-CLIP; F_cft_rep = LoRA-CLIP instead of frozen CLIP. R0_ref copied from nb39_cv.csv.
#          Stage A usual folds seeds 1-3; stage B NEW folds + seeds 4-6 (the only stage that counts). Training-side only; per-trial scoring.
SUMMARY = list(NOTES41) + SUMMARY
d = np.load(os.path.join(FEATDIR, 'clip_ft.npz')); FEAT['clip_ft'] = {s: d[s] for s in ['train', 'mav', 'extra']}
assert all(len(FEAT['clip_ft'][s]) == len(FEAT['clip'][s]) for s in ['train', 'mav', 'extra']), 'clip_ft rows do not match - tell Claude'
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def train_e(F_raw, V_raw, all_tr, vsets, seed, w_al=0.5, tau=0.07):
    """system E's recipe (= NB35 train_lout = NB39 train_e): 16 ids x 8 clips, CE + 2*OPL + 0.5*L_out, early stopping on the fold's pairs"""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
COMBOS = {'F_cft_add': (['clip', 'clip_ft'], VOICE), 'F_cft_rep': (['clip_ft'], VOICE)}
def inputs(fn_, vn_):
    F_raw = build_train(fn_); V_full = build_train(vn_); U, _ = nap_basis(V_full[N_CH:], NAP_K); return F_raw, nap_apply(V_full, U)
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV41 = os.path.join(RESDIR, 'nb41_cv.csv')
if not os.path.exists(CSV41):
    r = pd.read_csv(os.path.join(RESDIR, 'nb39_cv.csv')); r = r[r.combo == 'R0_ref'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    assert len(r) == 30, 'R0_ref incomplete in nb39_cv.csv - tell Claude'
    r.to_csv(CSV41, index=False); print('R0_ref copied from NB39 (30 runs)')
done41 = pd.read_csv(CSV41); t0 = time.time()
def run41(stage, names, seeds, fold_data):
    global done41
    for name in names:
        todo = [(seed, k) for seed in seeds for k in range(K) if not ((done41.stage == stage) & (done41.combo == name) & (done41.seed == seed) & (done41.fold == k)).any()]
        if not todo:
            continue
        F_raw, V_raw = inputs(*COMBOS[name])
        for seed in seeds:
            for k in range(K):
                if (seed, k) not in todo:
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_e(F_raw, V_raw, np.concatenate([tr_idx, EXT]), vsets, seed)
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV41, mode='a', header=False, index=False); done41 = pd.concat([done41, row])
            print('%s %-10s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
        del F_raw, V_raw
def tab41(stage):
    R = pd.read_csv(CSV41); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def adopt(TB, c):
    refB = TB.loc['R0_ref']; dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    return (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)), dB, ds, dg
run41('A', list(COMBOS), [1, 2, 3], FOLD_DATA)
run41('B', list(COMBOS), [4, 5, 6], FOLD_B)
TA, TB = tab41('A'), tab41('B')
note('NB41 STAGE A (usual folds, seeds 1-3; R0_ref = system E recipe):'); note(TA.to_string())
note('NB41 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs R0_ref):')
WIN = []
for c in COMBOS:
    ok, dB, ds, dg = adopt(TB, c)
    note('%-10s LoRA-CLIP %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, 'added' if c.endswith('add') else 'replacing frozen CLIP', TA.loc[c, 'overall'] - TA.loc['R0_ref', 'overall'], dB, ds, dg, 'ADOPT' if ok else 'reject'))
    if ok:
        WIN.append(c)
if WIN:
    note('NEXT: ADOPTED (%s). vox_<tag>_clip_ft.npy and final/clip_ft.pt are on Drive -> Claude builds the NB20 system J (dev upload, then eval).' % ', '.join(WIN))
else:
    note('NEXT: nothing adopted; the face side stays frozen CLIP.')
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
